<div style="text-align: center; background-color: #5A96E3; font-family: 'Trebuchet MS', Arial, sans-serif; color: white; padding: 20px; font-size: 40px; font-weight: bold; border-radius: 0 0 0 0; box-shadow: 0px 6px 8px rgba(0, 0, 0, 0.2);">
  Stage 04 - RAG implementation 📌
</div>

## I. Import libraries

In [1]:
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.embeddings import GPT4AllEmbeddings
from langchain.schema import HumanMessage
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

## II. Config environment

In [ ]:
os.environ["DEEPSEEK_API_KEY"] = ""

## III. RAG for question answering

In [3]:
llm = ChatDeepSeek(model="deepseek-chat")
embedding_model = GPT4AllEmbeddings()

In [ ]:
# client = QdrantClient(path="../qdrant_initial_db")

# collection_name = "candidates"

# vectorstore = Qdrant(
#     client=client,
#     collection_name=collection_name,
#     embeddings=embedding_model,
#     content_payload_key=None
# )

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_24248\2832832855.py:5: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use :class:`~QdrantVectorStore` instead.
  vectorstore = Qdrant(


In [ ]:
client = QdrantClient(path="../qdrant_db") 


In [5]:
vectorstore = Qdrant(
    client=client,
    collection_name="candidates",
    embeddings=embedding_model
)
retriever = vectorstore.as_retriever(search_kwargs={"k":3})
results = retriever.get_relevant_documents("3 Frontend React.js developers")

for doc in results:
    print(doc.metadata)
    print(doc.page_content)

{'name': 'Obi Nwokogba', 'email': 'obi.nwokogba@gmail.com', 'skills': ['JavaScript', 'TypeScript', 'Python', 'CSS3', 'HTML5', 'Java', 'PHP', 'SQL', 'Sass', 'MQL', 'Angular', 'React', 'React Native', 'HTMX', 'Express', 'NestJS', 'MongoDB', 'NodeJS', 'Responsive Design', 'Android App Development', 'UI design', 'Database Architecture', 'Git', 'Data Structures', 'Version Repository', 'Web APIs', 'Data Visualization', 'Agile Methodology', 'Front-End Web Development', 'Full-Stack Web Development', 'MySQL', 'Bootstrap', 'ERDs', 'Graphic Design', 'Algorithmic Trading', 'Financial Markets'], 'experience': 'Frontend Engineer at Madison Logic Inc. | Full Stack Engineer, Graphic Designer at Pregen Inc.', '_id': '438a7579820048b7b961abc86ffa6849', '_collection_name': 'candidates'}
Name: Obi Nwokogba, Email: obi.nwokogba@gmail.com, Skills: JavaScript, TypeScript, Python, CSS3, HTML5, Java, PHP, SQL, Sass, MQL, Angular, React, React Native, HTMX, Express, NestJS, MongoDB, NodeJS, Responsive Design, A

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6836\1477991691.py:1: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use :class:`~QdrantVectorStore` instead.
  vectorstore = Qdrant(
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6836\1477991691.py:7: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents("3 Frontend React.js developers")


In [20]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

template = """
Bạn là một hệ thống quản lý những ứng viên tài năng bằng AI. Bạn sẽ được truy cập các trường dữ liệu về ứng viên như:
Name: Tên ứng viên vừa ứng tuyển
Email: Email của họ.
Skill: Những kĩ năng của ứng viên như kĩ năng về kĩ thuật (như là ngôn ngữ lập trình, các frameworks) và các kĩ năng mềm (Problem Solving, Communication, Teamwork, ...).
Experience: Những kinh nghiệm làm việc của ứng viên, bao gồm các công ty đã làm việc, vị trí công việc, thời gian làm việc và mô tả công việc.

Bạn sẽ nhận được câu hỏi từ người dùng. Nhiệm vụ của bạn là:
1. Nếu là tiếng Việt hãy chuyển đổi câu hỏi sang tiếng Anh một cách ngắn gọn, rõ ràng, tập trung vào từ khóa quan trọng.
2. Sử dụng câu hỏi tiếng Anh để tìm kiếm thông tin ứng viên trong cơ sở dữ liệu. Ví dụ như từ câu hỏi tiếng Việt "5 ứng viên có Framework React.js trong phần kĩ năng?" sẽ được chuyển thành "5 candidates with React.js in skills?".
3. Trả lời câu hỏi dựa trên thông tin tìm được từ cơ sở dữ liệu.

Lưu ý: 
- Câu hỏi từ người dùng có thể liên quan đến các kỹ năng cụ thể, kinh nghiệm làm việc hoặc các thông tin khác về ứng viên.
- Trả lời cần ngắn gọn và rõ ràng, chỉ cung cấp thông tin cần thiết.

Câu hỏi gốc: {question}

Thông tin tìm được từ cơ sở dữ liệu:
{context}

Trả lời ngắn gọn và rõ ràng:
"""
QA_PROMPT = PromptTemplate(input_variables=["question", "context"], template=template)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": QA_PROMPT},
)

query = "Tất cả những ứng viên có Python trong phần Skills?"
result = qa_chain.run(query)

print("Kết quả:", result)

docs_with_scores = vectorstore.similarity_search_with_score(query, k=10)

with open("react_result2.txt", "w", encoding="utf-8") as f:
    f.write("=== Retrieved Documents with Cosine Similarity ===\n")
    f.write(query)
    for i, (doc, score) in enumerate(docs_with_scores, 1):
        f.write(f"\n--- Candidate {i} ---\n")
        f.write(f"Name: {doc.metadata.get('name')}\n")
        f.write(f"Email: {doc.metadata.get('email')}\n")
        f.write(f"Skills: {doc.metadata.get('skills')}\n")
        f.write(f"Experience: {doc.metadata.get('experience')}\n")
        f.write(f"Cosine similarity: {score:.4f}\n")
        f.write(f"Raw text: {doc.page_content}\n")


Kết quả: Here are the candidates with Python in their skills:

1. **Maheshwar Kuchana**  
   - Email: kmaheshwarl998@gmail.com  
   - Skills: Python, PyTorch, TensorFlow, Keras, OpenCV, Scikit-learn, Flask, AWS, etc.  
   - Experience: Data Engineer, AI/ML roles.  

2. **Pooya Karimian**  
   - Email: me@pooyak.com  
   - Skills: Python, C++, JavaScript, AWS, ML/AI, etc.  
   - Experience: Software Engineer at Meta, Facebook, etc.  

3. **Varsha L. Thirumalai**  
   - Email: vthiru300@gmail.com  
   - Skills: Python, C, Verilog, FPGA, etc.  
   - Experience: Software Engineer at Wipro (Ford Motors).  

4. **Vaibhav Kulkarni**  
   - Email: vkulkarn@protonmail.com  
   - Skills: Python, Java, Scala, Azure, ML, etc.  
   - Experience: Engineering Manager (Data/Bioinformatics).  

5. **Jake Mofa**  
   - Email: Jakemofa@gmail.com  
   - Skills: Python, Node.js, Django, AWS, etc.  
   - Experience: Software Engineer at VENDO.  

---  
**Total**: 5 candidates found.
